# 02a – Player Team Dashboard: Relación de métricas con Victoria/Impacto

Este cuaderno analiza el dashboard de jugadores por equipo para relacionar las métricas numéricas originales con los indicadores de victoria/derrota (`W`, `L`, `W_PCT`) y el impacto (`PLUS_MINUS`).

**Objetivos principales:**
- Explorar de forma sistemática cómo se comportan las métricas disponibles respecto a los objetivos de resultado e impacto.
- Detectar patrones generales, diferencias Home/Away y posibles incoherencias en columnas de rangos.
- Registrar hallazgos clave para orientar análisis predictivos posteriores sin crear métricas derivadas por jugador.

**Notas:**
- Solo se emplean las columnas originales; se permiten agregaciones para comparaciones (por ejemplo, medias por `GROUP_SET`).
- Las secciones incluyen comentarios que explican qué observar en cada visualización o tabla resultante.


## 1. Configuración e importaciones

Definimos las variables de configuración, importamos las librerías necesarias y establecemos estilos de visualización coherentes. Ajusta `CSV_PATH` con la ruta del archivo CSV a analizar antes de ejecutar el cuaderno.


In [104]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Variables de configuración (ajusta CSV_PATH según tu entorno)
CSV_PATH = Path("/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/dashboards/team_player_dashboard__dataset_1.parquet")
SAVE_FIG = True
FIG_DPI = 110
SEED = 42

np.random.seed(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)
plt.style.use("seaborn-v0_8")
sns.set_context("talk")
warnings.filterwarnings("ignore", category=FutureWarning)

OUTPUT_DIR = Path("outputs")
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

# Utilidades auxiliares

def ensure_dir(path: Path) -> Path:
    """Crea el directorio si no existe y devuelve la ruta."""
    path.mkdir(parents=True, exist_ok=True)
    return path


def safe_filename(name: str) -> str:
    """Genera nombres de archivo sencillos reemplazando caracteres problemáticos."""
    keep = [c if c.isalnum() or c in ("_", "-", " ") else "_" for c in name]
    cleaned = "".join(keep).strip().replace(" ", "_")
    return cleaned.lower()


def save_figure(fig: plt.Figure, path: Path) -> None:
    """Guarda una figura si SAVE_FIG es True."""
    if not SAVE_FIG:
        return
    ensure_dir(path.parent)
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")


def compute_spearman_correlations(df: pd.DataFrame, target: str, columns: list[str]) -> pd.DataFrame:
    """Calcula correlaciones de Spearman para las columnas numéricas respecto al objetivo indicado."""
    records = []
    for col in columns:
        if col == target:
            continue
        if col not in df.columns:
            continue
        series = df[[col, target]].dropna()
        if series.shape[0] < 5:
            continue
        if series[col].nunique(dropna=True) < 2 or series[target].nunique(dropna=True) < 2:
            continue
        rho = series[col].corr(series[target], method="spearman")
        if pd.notna(rho):
            records.append({
                "variable": col,
                "rho_spearman": rho,
                "n_observaciones": int(series.shape[0])
            })
    if not records:
        return pd.DataFrame(columns=["variable", "rho_spearman", "n_observaciones"])
    result = pd.DataFrame(records).sort_values("rho_spearman", ascending=False).reset_index(drop=True)
    return result



# Creación inicial de carpetas de salida
ensure_dir(FIGURES_DIR)
ensure_dir(TABLES_DIR)


PosixPath('outputs/tables')

## 2. Carga, limpieza ligera y tipado de columnas

Se carga el CSV indicado, se normalizan los nombres de columnas y se reconcilian duplicidades (`TEAM_ID`/`team_id`, `SEASON_YEAR`/`season`). Además, se definen los grupos de columnas categóricas, numéricas y de objetivos, convirtiendo las numéricas a formato adecuado. Finalmente se revisa la estructura general y los valores nulos más frecuentes.


In [105]:
if not CSV_PATH.exists():
    warnings.warn(f"⚠️ El archivo no existe en la ruta indicada: {CSV_PATH}")

df = pd.DataFrame()

if CSV_PATH.exists():
    try:
        df = pd.read_parquet(CSV_PATH)
        print(f"✅ Archivo parquet leído correctamente: {CSV_PATH}")
        print(f"Shape inicial: {df.shape}")
        print(f"Primeras columnas: {list(df.columns)[:15]}{'...' if len(df.columns) > 15 else ''}")
    except Exception as e:
        warnings.warn(f"❌ Error al leer el parquet: {e}")
        df = pd.DataFrame()

    # --- Limpieza de columnas ---
    df.columns = [c.strip() for c in df.columns]

    # --- Armonización TEAM_ID / team_id ---
    if "TEAM_ID" in df.columns and "team_id" in df.columns:
        df["TEAM_ID"] = df["TEAM_ID"].fillna(df["team_id"])
        df["team_id"] = df["team_id"].fillna(df["TEAM_ID"])
    elif "team_id" in df.columns and "TEAM_ID" not in df.columns:
        df["TEAM_ID"] = df["team_id"]
    elif "TEAM_ID" in df.columns and "team_id" not in df.columns:
        df["team_id"] = df["TEAM_ID"]

    # --- Armonización season / SEASON_YEAR ---
    if "season" in df.columns and "SEASON_YEAR" in df.columns:
        df["season"] = df["season"].fillna(df["SEASON_YEAR"])
        df["SEASON_YEAR"] = df["SEASON_YEAR"].fillna(df["season"])
    elif "SEASON_YEAR" in df.columns and "season" not in df.columns:
        df["season"] = df["SEASON_YEAR"]
    elif "season" in df.columns and "SEASON_YEAR" not in df.columns:
        df["SEASON_YEAR"] = df["season"]

    # --- Diagnóstico/filtrado de 'dataset' solo si aporta algo ---
    if "dataset" in df.columns:
        uniques = pd.Series(df["dataset"]).dropna().unique()
        print(f"Valores únicos en 'dataset': {sorted(uniques.tolist()) if len(uniques) else uniques}")
        # Solo filtramos si HAY valor 1 y además hay más de un valor distinto
        if (1 in uniques) and (len(uniques) > 1):
            before = df.shape
            df = df[df["dataset"].eq(1)].copy()
            print(f"📉 Filtrado dataset==1: {before} -> {df.shape}")
        else:
            print("ℹ️ No se filtra por 'dataset' (archivo ya corresponde a un dataset concreto o no hay valor 1).")

# --- Clasificación de columnas ---
cat_base = [
    "TEAM_ID",
    "TEAM_NAME",
    "PLAYER_ID",
    "PLAYER_NAME",
    "NICKNAME",
    "GROUP_SET",
    "season",
    "SEASON_YEAR",
    "season_type",
    "endpoint",
]
cat_cols = [c for c in cat_base if c in df.columns]

rank_cols = [c for c in df.columns if c.endswith("_RANK")]
target_cols = [c for c in ["W", "L", "W_PCT", "PLUS_MINUS"] if c in df.columns]

num_cols = [c for c in df.columns if c not in cat_cols]

# --- Conversión a numérico segura ---
for col in num_cols:
    if df.empty:
        break
    df[col] = pd.to_numeric(df[col], errors="coerce")

# --- Diagnóstico rápido ---
print(f"\nShape final del DataFrame: {df.shape}")
if not df.empty:
    display(df.head())
    na_summary = df.isna().sum().sort_values(ascending=False)
    display(na_summary.head(20).to_frame(name="n_nulos"))
    print(f"Columnas categóricas: {len(cat_cols)} | Numéricas: {len(num_cols)} | Objetivos: {target_cols}")
else:
    print("⚠️ El DataFrame está vacío. Revisa que el parquet no venga ya filtrado o sin filas.")

✅ Archivo parquet leído correctamente: /Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/dashboards/team_player_dashboard__dataset_1.parquet
Shape inicial: (950, 72)
Primeras columnas: ['TEAM_ID', 'TEAM_NAME', 'PLAYER_ID', 'GROUP_SET', 'PLAYER_NAME', 'NICKNAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M']...
Valores únicos en 'dataset': ['dataset_1']
ℹ️ No se filtra por 'dataset' (archivo ya corresponde a un dataset concreto o no hay valor 1).

Shape final del DataFrame: (950, 73)


,TEAM_ID,TEAM_NAME,PLAYER_ID,GROUP_SET,PLAYER_NAME,NICKNAME,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT,season,endpoint,team_id,season_type,dataset,SEASON_YEAR
0,1610612737,Atlanta Hawks,203991,Players,Clint Capela,Clint,55,27,28,0.491,1176.433333,218,390,0.559,0,1,0.000,52,97,0.536,174,295,469,62,47,34,53,36,106,89,488,-79,1357.8,11,0,1193.0,6,6,27,11,6,7,7,4,28,29,28,8,7,28,2,4,2,9,9,8,3,27,28,7,7,30,6,4,3,6,1,2024-25,team_player_dashboard,1610612737,Regular Season,NaN,2024-25
1,1610612737,Atlanta Hawks,203992,Players,Bogdan Bogdanović,Bogdan,24,12,12,0.500,597.878333,83,224,0.371,44,146,0.301,30,34,0.882,12,54,66,49,37,20,6,5,57,39,240,-44,433.7,0,0,451.0,17,16,15,4,15,14,12,26,10,9,21,12,13,5,16,17,17,13,10,13,16,14,20,12,14,23,16,14,3,15,1,2024-25,team_player_dashboard,1610612737,Regular Season,NaN,2024-25
2,1610612737,Atlanta Hawks,1626204,Players,Larry Nance Jr.,Larry,24,10,14,0.417,463.226667,80,155,0.516,34,76,0.447,9,13,0.692,23,80,103,38,16,20,13,6,37,25,203,-95,466.6,3,0,444.0,17,17,17,17,16,16,16,7,13,15,3,21,21,21,11,10,10,15,17,13,11,18,16,17,16,31,15,6,3,16,1,2024-25,team_player_dashboard,1610612737,Regular Season,NaN,2024-25
3,1610612737,Atlanta Hawks,1627747,Players,Caris LeVert,Caris,26,13,13,0.500,691.966667,145,301,0.482,46,136,0.338,52,72,0.722,19,77,96,75,25,24,14,12,51,52,388,20,704.7,1,0,681.0,16,14,16,4,10,8,8,12,9,11,18,8,9,17,13,11,11,8,13,12,10,24,18,9,9,6,9,8,3,9,1,2024-25,team_player_dashboard,1610612737,Regular Season,NaN,2024-25
4,1610612737,Atlanta Hawks,1627777,Players,Georges Niang,Georges,28,14,14,0.500,644.055000,120,272,0.441,76,184,0.413,23,29,0.793,13,70,83,44,34,11,7,5,72,28,339,-4,524.6,0,0,578.0,14,13,17,4,12,10,10,18,7,7,6,14,15,11,14,13,15,14,11,17,15,14,24,16,11,15,14,14,3,12,1,2024-25,team_player_dashboard,1610612737,Regular Season,NaN,2024-25


,n_nulos
dataset,950
NICKNAME,7
PLAYER_NAME,7
PLAYER_ID,0
TEAM_ID,0
GROUP_SET,0
GP,0
W,0
TEAM_NAME,0
L,0


Columnas categóricas: 10 | Numéricas: 63 | Objetivos: ['W', 'L', 'W_PCT', 'PLUS_MINUS']


## 3. Correlaciones globales con W_PCT y PLUS_MINUS

Calculamos correlaciones de Spearman entre cada métrica numérica y los objetivos principales. Las tablas ordenadas resaltan asociaciones fuertes; los gráficos de barras muestran los 20 valores más altos y más bajos por objetivo; y los mapas de calor permiten observar agrupaciones de métricas relacionadas con el rendimiento.


In [106]:
corr_tables = {}

if df.empty:
    warnings.warn("No hay datos cargados; se omiten correlaciones.")
else:
    corr_dir = ensure_dir(FIGURES_DIR / "corr")
    corr_table_dir = ensure_dir(TABLES_DIR / "corr")

    # --- 1) Excluir columnas fijas de las correlaciones ---
    explicit_exclude = {
        "team_id",
        "TEAM_COUNT",
        "W", "L",  # ⚠️ Excluye W y L de TODO (como features y también como posibles targets)
        "GP_RANK","W_RANK","L_RANK","W_PCT_RANK","MIN_RANK","FGM_RANK","FGA_RANK","FG_PCT_RANK",
        "FG3M_RANK","FG3A_RANK","FG3_PCT_RANK","FTM_RANK","FTA_RANK","FT_PCT_RANK",
        "OREB_RANK","DREB_RANK","REB_RANK","AST_RANK","TOV_RANK","STL_RANK","BLK_RANK","BLKA_RANK",
        "PF_RANK","PFD_RANK","PTS_RANK","PLUS_MINUS_RANK","NBA_FANTASY_PTS_RANK","DD2_RANK",
        "TD3_RANK","WNBA_FANTASY_PTS_RANK"
    }
    # Cubre cualquier *_RANK adicional no listado explícitamente
    auto_rank = {c for c in df.columns if c.upper().endswith("_RANK")}
    exclude_from_corr = explicit_exclude | auto_rank

    # --- 2) Targets válidos: quita W y L si venían en target_cols ---
    valid_targets = [t for t in target_cols if t in df.columns and t not in {"W", "L"}]

    # --- 3) Features numéricas permitidas (sin excluidas) ---
    numeric_candidates = [c for c in num_cols if c in df.columns and c not in exclude_from_corr]

    print(f"🎯 Targets seleccionados: {valid_targets}")
    print(f"🔎 Excluidas {len(exclude_from_corr)} columnas (team_id, TEAM_COUNT y *_RANK, W, L).")

    for target in valid_targets:
        # por seguridad, quita el propio target de la lista de features
        features = [c for c in numeric_candidates if c != target]

        corr_df = compute_spearman_correlations(df, target, features)
        if corr_df.empty:
            warnings.warn(f"Sin correlaciones válidas para {target}.")
            continue

        corr_tables[target] = corr_df
        csv_path = corr_table_dir / f"corr_spearman_{safe_filename(target)}.csv"
        corr_df.to_csv(csv_path, index=False)
        display(corr_df.head(10))

        # --- Barras: top y bottom por rho firmado ---
        top_n = corr_df.head(20)
        bottom_n = corr_df.tail(20)

        for subset, label in [(top_n, "top"), (bottom_n, "bottom")]:
            if subset.empty:
                continue
            fig, ax = plt.subplots(figsize=(10, max(6, 0.35 * len(subset))))
            sns.barplot(
                data=subset,
                y="variable",
                x="rho_spearman",
                palette="crest" if label == "top" else "flare",
                ax=ax,
            )
            ax.set_title(f"Correlaciones {label} con {target}")
            ax.set_xlabel("Rho de Spearman")
            ax.set_ylabel("Métrica")
            ax.axvline(0, color="gray", linestyle="--", linewidth=1)
            plt.tight_layout()
            save_figure(fig, corr_dir / f"{safe_filename(target)}_{safe_filename(label)}_20.png")
            plt.close(fig)

        # --- Heatmap: top 25 por |rho| ---
        top_abs = corr_df.reindex(corr_df["rho_spearman"].abs().sort_values(ascending=False).index).head(25)
        if not top_abs.empty:
            fig, ax = plt.subplots(figsize=(12, max(6, 0.4 * len(top_abs))))
            sns.heatmap(
                top_abs.set_index("variable")["rho_spearman"].to_frame(),
                annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax,
            )
            ax.set_title(f"Top 25 correlaciones (|rho|) con {target}")
            ax.set_xlabel("Rho de Spearman")
            plt.tight_layout()
            save_figure(fig, corr_dir / f"{safe_filename(target)}_heatmap_top25.png")
            plt.close(fig)


🎯 Targets seleccionados: ['W_PCT', 'PLUS_MINUS']
🔎 Excluidas 34 columnas (team_id, TEAM_COUNT y *_RANK, W, L).


,variable,rho_spearman,n_observaciones
0,PLUS_MINUS,0.491898,950
1,GP,0.112897,950
2,FG3_PCT,0.095061,950
3,STL,0.083792,950
4,NBA_FANTASY_PTS,0.076106,950
5,WNBA_FANTASY_PTS,0.073564,950
6,PF,0.071476,950
7,PTS,0.070343,950
8,FT_PCT,0.068479,950
9,PFD,0.068453,950


,variable,rho_spearman,n_observaciones
0,W_PCT,0.491898,950
1,FG_PCT,0.145383,950
2,FG3_PCT,0.115410,950
3,TD3,0.087503,950
4,DD2,0.035335,950
5,FT_PCT,0.022341,950
6,STL,0.017068,950
7,NBA_FANTASY_PTS,0.006292,950
8,WNBA_FANTASY_PTS,0.002496,950
9,FG3M,-0.001353,950


## 4. Relaciones directas: dispersión y densidad

Visualizamos cada métrica numérica frente a los objetivos (`W_PCT` y `PLUS_MINUS`). Los gráficos permiten detectar relaciones lineales o no lineales, además de dispersiones altas o grupos particulares. Se emplea `hexbin` cuando la cantidad de observaciones es elevada para mejorar la legibilidad.


In [107]:
# === Parámetros de selección de gráficos ===
TOP_N_PER_TARGET = 12  # cambia a 4, 10, etc.

def plot_metric_vs_target(metric: str, target: str) -> None:
    if metric not in df.columns or target not in df.columns or df.empty:
        return
    data = df[[metric, target]].dropna()
    # filtros de calidad mínima
    if data.shape[0] < 15 or data[metric].nunique() <= 3 or data[target].nunique() <= 3:
        return

    fig, ax = plt.subplots(figsize=(8, 6))
    if data.shape[0] > 500:
        hb = ax.hexbin(data[metric], data[target], gridsize=30, cmap="viridis", mincnt=1)
        cb = fig.colorbar(hb, ax=ax)
        cb.set_label("Densidad de observaciones")
    else:
        ax.scatter(data[metric], data[target], alpha=0.6, edgecolor="none", s=35)

    ax.set_title(f"{metric} vs {target}")
    ax.set_xlabel(metric)
    ax.set_ylabel(target)
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    out_dir = ensure_dir(FIGURES_DIR / "scatter" / safe_filename(target))
    out_path = out_dir / f"{safe_filename(metric)}_vs_{safe_filename(target)}.png"
    save_figure(fig, out_path)
    plt.close(fig)

# === Selección de métricas a graficar por target usando correlaciones existentes ===
if not df.empty and 'corr_tables' in locals() and corr_tables:
    total_plots = 0
    for target, corr_df in corr_tables.items():
        if corr_df.empty:
            continue
        # Top-N por |rho|
        top_vars = (
            corr_df.reindex(corr_df["rho_spearman"].abs().sort_values(ascending=False).index)
                  .head(TOP_N_PER_TARGET)["variable"]
                  .tolist()
        )
        print(f"🎯 {target}: graficando {len(top_vars)} métricas (top |rho|)")
        for metric in top_vars:
            plot_metric_vs_target(metric, target)
            total_plots += 1
    print(f"✅ Gráficos generados: {total_plots}")
else:
    warnings.warn("No hay corr_tables disponibles; genera correlaciones primero para seleccionar las métricas más informativas.")


🎯 W_PCT: graficando 12 métricas (top |rho|)
🎯 PLUS_MINUS: graficando 12 métricas (top |rho|)
✅ Gráficos generados: 24


## 6. Coherencia de columnas `_RANK`

Se comprueba si los rangos (`*_RANK`) mantienen coherencia con su métrica base: esperamos correlaciones negativas (a menor rango, mejor valor). Además, se visualizan las distribuciones de `W_PCT` por cuartiles del rango para detectar inconsistencias.


In [108]:
rank_results = []
if df.empty or not rank_cols:
    warnings.warn("No hay columnas _RANK disponibles para evaluar.")
else:
    ranks_dir = ensure_dir(FIGURES_DIR / "ranks")
    ranks_table_dir = ensure_dir(TABLES_DIR / "ranks")
    for rank_col in rank_cols:
        metric = rank_col.replace("_RANK", "")
        if metric not in df.columns:
            continue
        data = df[[metric, rank_col]].dropna()
        if data.shape[0] < 10 or data[metric].nunique() < 3 or data[rank_col].nunique() < 3:
            continue
        rho = data[metric].corr(data[rank_col], method="spearman")
        rank_results.append({
            "rank_col": rank_col,
            "metric": metric,
            "rho_spearman": rho,
            "n_observaciones": int(data.shape[0]),
        })

        fig, ax = plt.subplots(figsize=(6, 5))
        sns.scatterplot(data=data, x=metric, y=rank_col, alpha=0.7, ax=ax)
        ax.set_title(f"Coherencia {metric} vs {rank_col} (rho={rho:.2f})")
        ax.set_xlabel(metric)
        ax.set_ylabel(rank_col)
        plt.tight_layout()
        save_figure(fig, ranks_dir / f"scatter_{safe_filename(metric)}_{safe_filename(rank_col)}.png")
        plt.close(fig)

        if "W_PCT" in df.columns:
            temp = df[[rank_col, "W_PCT"]].copy().dropna()
            if not temp.empty:
                # Asegura numérico y elimina NaN antes de qcut
                rnum = pd.to_numeric(temp[rank_col], errors="coerce")
                valid = rnum.notna()
                temp = temp.loc[valid]
                rnum = rnum.loc[valid]

                # qcut con bins colapsables; luego renombramos según nº real de bins
                qseries = pd.qcut(rnum, q=4, duplicates="drop")
                labels = [f"Q{i}" for i in range(1, len(qseries.cat.categories) + 1)]
                quartiles = qseries.cat.rename_categories(labels)

                # Boxplot por quantiles del rank
                fig, ax = plt.subplots(figsize=(6, 5))
                sns.boxplot(x=quartiles, y=temp["W_PCT"], palette="Blues", ax=ax)
                ax.set_title(f"W_PCT por quantiles de {rank_col}")
                ax.set_xlabel("Quantil del rango")
                ax.set_ylabel("W_PCT")
                plt.tight_layout()
                save_figure(fig, ranks_dir / f"boxplot_wpct_{safe_filename(rank_col)}.png")
                plt.close(fig)

    if rank_results:
        rank_df = pd.DataFrame(rank_results).sort_values("rho_spearman")
        (ranks_table_dir / "coherencia_ranks.csv").write_text(rank_df.to_csv(index=False))
        display(rank_df.head(10))


,rank_col,metric,rho_spearman,n_observaciones
9,FG3A_RANK,FG3A,-0.967812,950
6,FGA_RANK,FGA,-0.967411,950
26,NBA_FANTASY_PTS_RANK,NBA_FANTASY_PTS,-0.967097,950
24,PTS_RANK,PTS,-0.966831,950
29,WNBA_FANTASY_PTS_RANK,WNBA_FANTASY_PTS,-0.966673,950
0,GP_RANK,GP,-0.966590,950
16,REB_RANK,REB,-0.966588,950
15,DREB_RANK,DREB,-0.966286,950
4,MIN_RANK,MIN,-0.965990,950
5,FGM_RANK,FGM,-0.965726,950


## 7. Relación eficiencia vs volumen

Revisamos pares de métricas que combinan porcentajes con volumen (intentos o totales) para identificar zonas de mayor eficacia. Los colores reflejan `W_PCT` o `PLUS_MINUS`, facilitando detectar jugadores con impacto positivo.


In [109]:
if df.empty:
    warnings.warn("Sin datos para evaluar eficiencia vs volumen.")
else:
    eff_dir = ensure_dir(FIGURES_DIR / "efficiency_volume")
    pairs = [
        ("FG_PCT", "FGA"),
        ("FG3_PCT", "FG3A"),
        ("FT_PCT", "FTA"),
        ("AST", "TOV"),
        ("OREB", "DREB"),
    ]
    color_targets = [t for t in ["W_PCT", "PLUS_MINUS"] if t in df.columns]
    for x, y in pairs:
        if x not in df.columns or y not in df.columns:
            continue
        data = df[[x, y] + color_targets].dropna()
        if data.shape[0] < 20:
            continue
        for target in color_targets:
            fig, ax = plt.subplots(figsize=(7, 6))
            scatter = ax.scatter(
                data[x],
                data[y],
                c=data[target],
                cmap="viridis",
                alpha=0.7,
                edgecolor="none",
            )
            ax.set_title(f"{x} vs {y} coloreado por {target}")
            ax.set_xlabel(x)
            ax.set_ylabel(y)
            cbar = plt.colorbar(scatter, ax=ax)
            cbar.set_label(target)
            ax.grid(True, linestyle="--", alpha=0.4)
            plt.tight_layout()
            save_figure(fig, eff_dir / f"{safe_filename(x)}_vs_{safe_filename(y)}_{safe_filename(target)}.png")
            plt.close(fig)


## 8. Defensa y control de faltas

Analizamos cómo las acciones defensivas (`STL`, `BLK`) y las faltas (`PF`, `BLKA`) se relacionan con los objetivos. Se incluyen diagramas de dispersión/hexbin y un mini mapa de calor con las correlaciones disponibles.


In [110]:
defense_dir = ensure_dir(FIGURES_DIR / "defense")
def_metrics = ["STL", "BLK", "PF", "BLKA"]
def_targets = [t for t in ["W_PCT", "PLUS_MINUS"] if t in df.columns]

for metric in def_metrics:
    if metric not in df.columns:
        continue
    for target in def_targets:
        plot_metric_vs_target(metric, target)

if def_targets:
    corr_data = []
    for metric in def_metrics:
        if metric not in df.columns:
            continue
        for target in def_targets:
            valid = df[[metric, target]].dropna()
            if valid.shape[0] < 5:
                continue
            rho = valid[metric].corr(valid[target], method="spearman")
            corr_data.append({"Métrica": metric, "Objetivo": target, "rho_spearman": rho})
    if corr_data:
        corr_df = pd.DataFrame(corr_data)
        pivot = corr_df.pivot(index="Métrica", columns="Objetivo", values="rho_spearman")
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.heatmap(pivot, annot=True, cmap="coolwarm", center=0, fmt=".2f", ax=ax)
        ax.set_title("Correlaciones defensa/faltas vs objetivos")
        plt.tight_layout()
        save_figure(fig, defense_dir / "correlaciones_defensa.png")
        plt.close(fig)


## 9. Perspectiva por equipo y jugador

Se analizan diferencias entre equipos y jugadores: boxplots de `W_PCT` por equipo, barras de `PLUS_MINUS` por jugador dentro de cada equipo (coloreadas por minutos), y detección de outliers interesantes para profundizar en futuros análisis.


In [111]:
if {"PLUS_MINUS", "GP"}.issubset(df.columns):
    df["PLUS_MINUS_PER_GAME"] = df["PLUS_MINUS"] / df["GP"].replace(0, np.nan)
    print("✅ Añadida columna PLUS_MINUS_PER_GAME (PLUS_MINUS / GP)")
else:
    warnings.warn("⚠️ No se pudo crear PLUS_MINUS_PER_GAME (faltan PLUS_MINUS o GP).")

team_dir = ensure_dir(FIGURES_DIR / "teams")

# 2) Boxplot de W_PCT por equipo (igual que antes)
if "TEAM_NAME" in df.columns and "W_PCT" in df.columns:
    data = df[["TEAM_NAME", "W_PCT"]].dropna()
    if data["TEAM_NAME"].nunique() > 1:
        fig, ax = plt.subplots(figsize=(max(10, data["TEAM_NAME"].nunique() * 0.6), 6))
        sns.boxplot(data=data, x="TEAM_NAME", y="W_PCT", palette="Set3", ax=ax)
        ax.set_title("Distribución de W_PCT por equipo")
        ax.set_xlabel("Equipo")
        ax.set_ylabel("W_PCT")
        ax.tick_params(axis="x", labelrotation=45)  # rotación correcta
        plt.setp(ax.get_xticklabels(), ha="right")  # alineación horizontal
        plt.tight_layout()
        save_figure(fig, team_dir / "boxplot_wpct_por_equipo.png")
        plt.close(fig)

# 3) Barras de PLUS_MINUS por partido por jugador y equipo
if {"TEAM_NAME", "PLAYER_NAME"}.issubset(df.columns) and "PLUS_MINUS_PER_GAME" in df.columns:
    pivot_cols = [c for c in ["MIN"] if c in df.columns]
    for team, team_df in df.groupby("TEAM_NAME", dropna=True):
        team_df = team_df.dropna(subset=["PLAYER_NAME", "PLUS_MINUS_PER_GAME"])
        if team_df.empty:
            continue
        team_df = team_df.sort_values("PLUS_MINUS_PER_GAME", ascending=False)

        fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(team_df))))

        if "MIN" in pivot_cols and team_df["MIN"].nunique() > 1:
            norm = plt.Normalize(team_df["MIN"].min(), team_df["MIN"].max())
            cmap = plt.cm.viridis
            colors = [cmap(norm(v)) for v in team_df["MIN"]]
            sns.barplot(x="PLUS_MINUS_PER_GAME", y="PLAYER_NAME", data=team_df, palette=colors, ax=ax)
            sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
            sm.set_array([])
            cbar = fig.colorbar(sm, ax=ax)
            cbar.set_label("MIN")
        else:
            sns.barplot(x="PLUS_MINUS_PER_GAME", y="PLAYER_NAME", data=team_df, palette="Blues", ax=ax)

        ax.set_title(f"PLUS_MINUS por partido – {team}")
        ax.set_xlabel("PLUS_MINUS por partido")
        ax.set_ylabel("Jugador")
        plt.tight_layout()
        save_figure(fig, team_dir / f"{safe_filename(team)}_players_plusminus_per_game.png")
        plt.close(fig)

# 4) Tablas de “insights”
ensure_dir(TABLES_DIR)

# Jugadores con muchos puntos pero equipos con W_PCT bajo (top 10)
if {"PLAYER_NAME", "TEAM_NAME", "PTS", "W_PCT"}.issubset(df.columns):
    temp = df[["PLAYER_NAME", "TEAM_NAME", "PTS", "W_PCT"]].dropna()
    if not temp.empty:
        temp = temp.sort_values(["PTS", "W_PCT"], ascending=[False, True]).head(10)
        (TABLES_DIR / "insights_pts_alto_wpct_bajo.csv").write_text(temp.to_csv(index=False))
        display(temp)

# Jugadores con muchos minutos pero PLUS_MINUS por partido bajo (top 10)
if {"PLAYER_NAME", "TEAM_NAME", "MIN", "PLUS_MINUS_PER_GAME"}.issubset(df.columns):
    temp = df[["PLAYER_NAME", "TEAM_NAME", "MIN", "PLUS_MINUS_PER_GAME"]].dropna()
    if not temp.empty:
        temp = temp.sort_values(["MIN", "PLUS_MINUS_PER_GAME"], ascending=[False, True]).head(10)
        (TABLES_DIR / "insights_min_alto_plusminus_per_game_bajo.csv").write_text(temp.to_csv(index=False))
        display(temp)
else:
    warnings.warn("⚠️ No se generó la tabla de MIN alto con PLUS_MINUS por partido bajo (faltan columnas).")

✅ Añadida columna PLUS_MINUS_PER_GAME (PLUS_MINUS / GP)


/var/folders/ws/b19hmrms3gdgnw9_l_qlf_qw0000gn/T/ipykernel_39513/886511026.py:39: UserWarning: The palette list has more values (32) than needed (31), which may not be intended.
  sns.barplot(x="PLUS_MINUS_PER_GAME", y="PLAYER_NAME", data=team_df, palette=colors, ax=ax)


,PLAYER_NAME,TEAM_NAME,PTS,W_PCT
732,Shai Gilgeous-Alexander,Oklahoma City Thunder,2484,0.829
414,Anthony Edwards,Minnesota Timberwolves,2177,0.608
191,Nikola Jokić,Denver Nuggets,2071,0.657
376,Giannis Antetokounmpo,Milwaukee Bucks,2036,0.597
36,Jayson Tatum,Boston Celtics,1932,0.736
603,Devin Booker,Phoenix Suns,1923,0.467
5,Trae Young,Atlanta Hawks,1841,0.474
352,Tyler Herro,Miami Heat,1840,0.481
884,Cade Cunningham,Detroit Pistons,1830,0.543
274,James Harden,LA Clippers,1802,0.633


,PLAYER_NAME,TEAM_NAME,MIN,PLUS_MINUS_PER_GAME
470,Mikal Bridges,New York Knicks,3036.076667,4.073171
469,Josh Hart,New York Knicks,2896.830000,2.636364
414,Anthony Edwards,Minnesota Timberwolves,2871.058333,3.683544
603,Devin Booker,Phoenix Suns,2794.966667,-2.360000
274,James Harden,LA Clippers,2789.036667,4.354430
658,DeMar DeRozan,Sacramento Kings,2767.615000,1.428571
5,Trae Young,Atlanta Hawks,2739.060000,0.447368
352,Tyler Herro,Miami Heat,2725.080000,1.285714
468,OG Anunoby,New York Knicks,2705.776667,4.067568
255,Jalen Green,Houston Rockets,2697.110000,2.024390


## 10. Fantasía y logros individuales

Se evalúa cómo los puntos de fantasía (`NBA_FANTASY_PTS`, `WNBA_FANTASY_PTS`) y los logros (`DD2`, `TD3`) se asocian con los objetivos. Se incluyen correlaciones y visualizaciones con bins de `W_PCT` para identificar tendencias.


In [112]:
fantasy_dir = ensure_dir(FIGURES_DIR / "fantasy")
fantasy_tables = []

fantasy_metrics = [c for c in ["NBA_FANTASY_PTS", "WNBA_FANTASY_PTS"] if c in df.columns]
fantasy_targets = [t for t in ["W_PCT", "PLUS_MINUS"] if t in df.columns]

for metric in fantasy_metrics:
    for target in fantasy_targets:
        data = df[[metric, target]].dropna()
        if data.shape[0] < 15:
            continue
        rho = data[metric].corr(data[target], method="spearman")
        fantasy_tables.append({"metric": metric, "objetivo": target, "rho_spearman": rho, "n": int(data.shape[0])})
        fig, ax = plt.subplots(figsize=(7, 6))
        if data.shape[0] > 500:
            hb = ax.hexbin(data[metric], data[target], gridsize=30, cmap="magma", mincnt=1)
            cbar = fig.colorbar(hb, ax=ax)
            cbar.set_label("Densidad")
        else:
            ax.scatter(data[metric], data[target], alpha=0.6, color="#d62728", edgecolor="none")
        ax.set_title(f"{metric} vs {target} (rho={rho:.2f})")
        ax.set_xlabel(metric)
        ax.set_ylabel(target)
        plt.tight_layout()
        save_figure(fig, fantasy_dir / f"{safe_filename(metric)}_vs_{safe_filename(target)}.png")
        plt.close(fig)

for binary in ["DD2", "TD3"]:
    if binary not in df.columns or "W_PCT" not in df.columns:
        continue
    data = df[[binary, "W_PCT"]].dropna()
    if data.empty or data[binary].nunique() <= 1:
        continue
    bins = np.linspace(data["W_PCT"].min(), data["W_PCT"].max(), 6)
    if len(np.unique(bins)) < 2:
        continue
    labels = [f"Bin {i+1}" for i in range(len(bins) - 1)]
    bin_assign = pd.cut(data["W_PCT"], bins=bins, labels=labels, include_lowest=True, duplicates="drop")
    prop = data.groupby(bin_assign)[binary].mean().dropna()
    if not prop.empty:
        fig, ax = plt.subplots(figsize=(7, 5))
        sns.barplot(x=prop.index, y=prop.values, palette="crest", ax=ax)
        ax.set_title(f"Proporción de {binary}=1 por tramo de W_PCT")
        ax.set_xlabel("Tramo de W_PCT")
        ax.set_ylabel(f"% {binary}=1")
        ax.set_ylim(0, 1)
        plt.tight_layout()
        save_figure(fig, fantasy_dir / f"proporcion_{safe_filename(binary)}.png")
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.stripplot(x=binary, y="W_PCT", data=data, jitter=True, alpha=0.5, ax=ax)
    ax.set_title(f"{binary} vs W_PCT")
    ax.set_xlabel(binary)
    ax.set_ylabel("W_PCT")
    plt.tight_layout()
    save_figure(fig, fantasy_dir / f"strip_{safe_filename(binary)}.png")
    plt.close(fig)

if fantasy_tables:
    fantasy_df = pd.DataFrame(fantasy_tables)
    ensure_dir(TABLES_DIR / "fantasy").joinpath("fantasy_relations.csv").write_text(fantasy_df.to_csv(index=False))
    display(fantasy_df)


,metric,objetivo,rho_spearman,n
0,NBA_FANTASY_PTS,W_PCT,0.076106,950
1,NBA_FANTASY_PTS,PLUS_MINUS,0.006292,950
2,WNBA_FANTASY_PTS,W_PCT,0.073564,950
3,WNBA_FANTASY_PTS,PLUS_MINUS,0.002496,950


## 11. Multicolinealidad entre predictores

Se inspecciona la correlación entre las métricas numéricas (excluyendo los objetivos) para detectar redundancias. Se genera un mapa de calor completo y otro reducido con las 40 columnas de mayor varianza cuando es necesario.


In [113]:
multicol_dir = ensure_dir(FIGURES_DIR / "multicollinearity")
if df.empty:
    warnings.warn("No se puede evaluar la multicolinealidad sin datos.")
else:
    predictors = [c for c in num_cols if c in df.columns and c not in target_cols]
    data = df[predictors].dropna(axis=1, how="all")
    if data.empty or data.shape[1] < 2:
        warnings.warn("Columnas numéricas insuficientes para correlaciones.")
    else:
        corr_matrix = data.corr(method="spearman")
        fig, ax = plt.subplots(figsize=(max(12, 0.4 * corr_matrix.shape[1]), max(10, 0.4 * corr_matrix.shape[0])))
        sns.heatmap(corr_matrix, cmap="coolwarm", center=0, ax=ax)
        ax.set_title("Correlación Spearman entre predictores")
        plt.tight_layout()
        save_figure(fig, multicol_dir / "heatmap_completo.png")
        plt.close(fig)

        variances = data.var().sort_values(ascending=False)
        top_cols = variances.head(40).index
        if len(top_cols) > 2 and len(top_cols) < len(predictors):
            corr_top = data[top_cols].corr(method="spearman")
            fig, ax = plt.subplots(figsize=(max(12, 0.4 * len(top_cols)), max(10, 0.4 * len(top_cols))))
            sns.heatmap(corr_top, cmap="coolwarm", center=0, ax=ax)
            ax.set_title("Correlación Spearman (Top 40 varianza)")
            plt.tight_layout()
            save_figure(fig, multicol_dir / "heatmap_top40.png")
            plt.close(fig)


## 12. Resumen ejecutivo

- **Drivers positivos de W_PCT:** _Completar tras la ejecución_ (revisar tablas de correlación y gráficas de eficiencia).
- **Drivers negativos de W_PCT:** _Completar tras la ejecución_ (considerar métricas con correlación negativa y diferencias Home/Away).
- **Variables con gran diferencia Home–Away:** _Completar revisando `home_away_deltas.csv` y los gráficos de barras_.
- **Outliers relevantes:** _Anotar jugadores/equipos detectados en las tablas de insights (`PTS` alto con `W_PCT` bajo, `MIN` alto con `PLUS_MINUS` bajo) y en los gráficos de dispersión_.

> Este espacio sirve para documentar hallazgos clave luego de inspeccionar las visualizaciones y tablas generadas.


## 13. Guardado final y reproducibilidad

Se verifica la creación de carpetas de salida, se contabilizan archivos exportados y se registran versiones de librerías y metadatos básicos de ejecución.


In [114]:
from datetime import datetime

figure_count = sum(len(files) for _, _, files in os.walk(FIGURES_DIR)) if FIGURES_DIR.exists() else 0
table_count = sum(len(files) for _, _, files in os.walk(TABLES_DIR)) if TABLES_DIR.exists() else 0

print(f"Figuras guardadas: {figure_count}")
print(f"Tablas guardadas: {table_count}")
print(f"Directorios de salida: {FIGURES_DIR.resolve()} | {TABLES_DIR.resolve()}")

print("Versiones de librerías clave:")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"matplotlib: {plt.matplotlib.__version__}")
print(f"seaborn: {sns.__version__}")

print(f"Semilla utilizada: {SEED}")
print(f"Fecha/Hora de ejecución: {datetime.now().isoformat(timespec='seconds')}")


Figuras guardadas: 180
Tablas guardadas: 11
Directorios de salida: /Users/pablo/Documents/BigData/BasketballAnalysis/02_processing_data/02a_WL_prediction/02a_params/outputs/figures | /Users/pablo/Documents/BigData/BasketballAnalysis/02_processing_data/02a_WL_prediction/02a_params/outputs/tables
Versiones de librerías clave:
pandas: 2.3.2
numpy: 2.3.3
matplotlib: 3.10.6
seaborn: 0.13.2
Semilla utilizada: 42
Fecha/Hora de ejecución: 2025-10-09T13:28:55
